# `06-workflow.ipynb`

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

llm = init_chat_model('gpt-4.1-mini')

## Prompt Chaining

- 매우 잘 정리된 업무 순서가 있을 경우 사용
- 이전 노드에서 처리한 내용을 `state` 에 담아 다음 노드로 전송

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 그림 보는 용
from IPython.display import Image, display


# Graph State
class State(TypedDict):
    topic: str
    joke: str
    improved_joke: str
    final_joke: str


# Nodes
def generate_joke(state: State):
    msg = llm.invoke(f'주제 {state['topic']}에 관련된 짧은 농담 생성')
    return {'joke': msg.content}


def improve_joke(state: State):
    msg = llm.invoke(f'말장난을 추가해서 아래 농담을 더 재밌게 만들어보자.\n {state['joke']}')
    return {'improved_joke': msg.content}


def polish_joke(state: State):
    msg = llm.invoke(f'아래 농담을 쩔게 뒤틀어 보자. \n {state['improved_joke']}')
    return {'final_joke': msg.content}


# Router
def check_punchline(state: State):
    # ? 나 ! 없으면, 농담을 improve 하고, 있으면 그대로 진행
    if '?' in state['joke'] or '!' in state['joke']:
        return 'Pass'
    else:
        return 'Fail'
    



AIMessage(content='봄에 관한 짧은 농담 하나 드릴게요!\n\n"봄이 와서 꽃들이 활짝 웃는데, 왜 벌들이 항상 바쁜 줄 알아? 그냥 \'꽃길만 걷자\'고 말하는 중이래!" 🌸🐝😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 20, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_3d19af3832', 'id': 'chatcmpl-DIOG09buSSb5C7tL6hXE0D0k0jVzY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cdf75-1baa-7873-af26-aa8bf3785574-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 59, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})